In [1]:
from transformers import BartTokenizer, BartModel
import torch

tokenizer = BartTokenizer.from_pretrained('facebook/bart-large')

model = BartModel.from_pretrained(
    'facebook/bart-large', 
    force_download=True
)

inputs = tokenizer("Hello, my dog is cute", return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

last_hidden_states = outputs.last_hidden_state
print("model shape:", last_hidden_states.shape)

config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

model shape: torch.Size([1, 8, 1024])


In [2]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
from datasets import Dataset
from transformers import (
    BartTokenizer, 
    BartForConditionalGeneration, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)

print("library load")

library load


In [3]:
pip install transformers

Note: you may need to restart the kernel to use updated packages.


In [4]:
# dataset check
DATASET_NAME = "XSAMSum"   # you can change this "XMediaSum40k"
MODEL_NAME = "facebook/bart-large"
CACHE_DIR = "./hf_cache"
BASE_DIR = f"../../data/raw/{DATASET_NAME}"

In [5]:
# dataset load
import os
from datasets import load_dataset

BASE_DIR = f"../../data/raw/{DATASET_NAME}"

data_files = {
    "train": os.path.join(BASE_DIR, "train.json"),
    "validation": os.path.join(BASE_DIR, "val.json"),
    "test": os.path.join(BASE_DIR, "test.json"),
}

print(data_files)

dataset_dict = load_dataset("json", data_files=data_files)
dataset_dict

{'train': '../../data/raw/XSAMSum/train.json', 'validation': '../../data/raw/XSAMSum/val.json', 'test': '../../data/raw/XSAMSum/test.json'}


DatasetDict({
    train: Dataset({
        features: ['dialogue', 'summary', 'summary_de', 'summary_zh'],
        num_rows: 14732
    })
    validation: Dataset({
        features: ['dialogue', 'summary', 'summary_de', 'summary_zh'],
        num_rows: 818
    })
    test: Dataset({
        features: ['dialogue', 'summary', 'summary_de', 'summary_zh'],
        num_rows: 819
    })
})

In [6]:
import pandas as pd

# DataFrame
df_train = dataset_dict['train'].to_pandas()

# result
display(df_train.head())

,dialogue,summary,summary_de,summary_zh
0,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...,amanda hat kekse gebacken und wird jerry morge...,阿曼达烤了饼干，明天会给杰瑞带一些。
1,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...,olivia und olivier stimmen bei dieser wahl für...,奥利维亚和奥利维尔在这次选举中要投票给自由党。
2,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...,kim kann die von tim empfohlene pomodoro-techn...,金可能会尝试蒂姆推荐的番茄工作法来完成更多的工作。
3,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...,"edward denkt, dass er in bella verliebt ist. r...",爱德华认为他爱上了贝拉。瑞秋想让爱德华开门。瑞秋在外面。
4,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com...","sam ist verwirrt, weil er mitbekommen hat, wie...",山姆很困惑，因为他无意中听到瑞克抱怨他这个室友。娜欧米觉得山姆应该和瑞克谈谈。山姆不知道该怎么办。


In [7]:
import pandas as pd

# convert into DataFrame
df_train = dataset_dict['train'].to_pandas()
df_val = dataset_dict['validation'].to_pandas()
df_test = dataset_dict['test'].to_pandas()

# result check
print(f"Train rows: {len(df_train)}")
print(f"Validation rows: {len(df_val)}")
print(f"Test rows: {len(df_test)}")

Train rows: 14732
Validation rows: 818
Test rows: 819


In [8]:
display(df_train.head())
display(df_val.head())
display(df_test.head())

,dialogue,summary,summary_de,summary_zh
0,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...,amanda hat kekse gebacken und wird jerry morge...,阿曼达烤了饼干，明天会给杰瑞带一些。
1,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...,olivia und olivier stimmen bei dieser wahl für...,奥利维亚和奥利维尔在这次选举中要投票给自由党。
2,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...,kim kann die von tim empfohlene pomodoro-techn...,金可能会尝试蒂姆推荐的番茄工作法来完成更多的工作。
3,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...,"edward denkt, dass er in bella verliebt ist. r...",爱德华认为他爱上了贝拉。瑞秋想让爱德华开门。瑞秋在外面。
4,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com...","sam ist verwirrt, weil er mitbekommen hat, wie...",山姆很困惑，因为他无意中听到瑞克抱怨他这个室友。娜欧米觉得山姆应该和瑞克谈谈。山姆不知道该怎么办。


,dialogue,summary,summary_de,summary_zh
0,"A: Hi Tom, are you busy tomorrow’s afternoon?\...",A will go to the animal shelter tomorrow to ge...,"a wird morgen ins tierheim gehen, um einen wel...",a明天要去动物收容所给她儿子买一只小狗。他们上周一已经去了收容所，她儿子选了这只小狗。
1,Emma: I’ve just fallen in love with this adven...,Emma and Rob love the advent calendar. Lauren ...,emma und rob lieben den adventskalender. laure...,艾玛和罗伯都很喜欢这个基督降临历。劳伦在日历内放各种物品，譬如小玩具和圣诞装饰品。她的孩子一...
2,Jackie: Madison is pregnant\r\nJackie: but she...,Madison is pregnant but she doesn't want to ta...,"madison ist schwanger, aber sie möchte darüber...",麦迪逊怀孕了，但她不想提及。帕特丽夏·史蒂文斯结婚了，她认为在这之前自己就怀孕了。
3,Marla: <file_photo>\r\nMarla: look what I foun...,Marla found a pair of boxers under her bed.,marla fand unter ihrem bett ein paar unterhosen.,玛拉在床底下发现了两条四角裤。
4,Robert: Hey give me the address of this music ...,Robert wants Fred to send him the address of t...,"robert möchte, dass fred ihm die adresse des m...",罗伯特想让弗雷德把音像店的地址发给他，因为他需要买吉他弦。


,dialogue,summary,summary_de,summary_zh
0,"Hannah: Hey, do you have Betty's number?\nAman...",Hannah needs Betty's number but Amanda doesn't...,"hannah braucht bettys nummer, aber amanda hat ...",汉娜需要贝蒂的电话号码，但阿曼达没有。她得联系拉里。
1,Eric: MACHINE!\r\nRob: That's so gr8!\r\nEric:...,Eric and Rob are going to watch a stand-up on ...,eric und rob werden sich ein stand-up auf yout...,埃里克和罗伯要在youtube上看一场单口相声。
2,"Lenny: Babe, can you help me with something?\r...",Lenny can't decide which trousers to buy. Bob ...,"lenny kann sich nicht entscheiden, welche hose...",莱尼无法决定买哪条裤子。鲍勃就此给莱尼提了些建议。莱尼听了他的建议，选了质量最好的裤子。
3,"Will: hey babe, what do you want for dinner to...",Emma will be home soon and she will let Will k...,emma kommt bald nach hause sein und sie wird e...,艾玛很快就会回家，而且她会告诉威尔。
4,"Ollie: Hi , are you in Warsaw\r\nJane: yes, ju...",Jane is in Warsaw. Ollie and Jane has a party....,jane ist in warschau. ollie und jane haben ein...,简在华沙，她和奥利有个聚会。她把重要的日子忘了，本来他们周五会共进午餐。但是奥利无意间给简打...


In [9]:
# 모든 split을 한 번에 변환하여 dfs 딕셔너리에 저장
dfs = {split: dataset.to_pandas() for split, dataset in dataset_dict.items()}

# 이제 아래와 같이 접근할 수 있습니다.
# dfs['train'], dfs['validation'], dfs['test']
display(dfs['train'].head())

,dialogue,summary,summary_de,summary_zh
0,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...,amanda hat kekse gebacken und wird jerry morge...,阿曼达烤了饼干，明天会给杰瑞带一些。
1,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...,olivia und olivier stimmen bei dieser wahl für...,奥利维亚和奥利维尔在这次选举中要投票给自由党。
2,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...,kim kann die von tim empfohlene pomodoro-techn...,金可能会尝试蒂姆推荐的番茄工作法来完成更多的工作。
3,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...,"edward denkt, dass er in bella verliebt ist. r...",爱德华认为他爱上了贝拉。瑞秋想让爱德华开门。瑞秋在外面。
4,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com...","sam ist verwirrt, weil er mitbekommen hat, wie...",山姆很困惑，因为他无意中听到瑞克抱怨他这个室友。娜欧米觉得山姆应该和瑞克谈谈。山姆不知道该怎么办。


In [10]:
# pick random sample
sample = df_train.sample(1)
print(f"대화 원문: {sample['dialogue'].values[0]}")
print(f"영어 요약: {sample['summary'].values[0]}")
print(f"독어 요약: {sample['summary_de'].values[0]}")
print(f"중어 요약: {sample['summary_zh'].values[0]}")

대화 원문: Lewis: Did you read today that Chrissy doesn't allow feet in her pictures?
Georgia: I did! How crazy!
Lewis: We all have our weird stuff but geez!
Georgia: I know, she made them retouch her and everything!
Lewis: I know! No idea why!
Georgia: I guess she just hates her feet.
Lewis: Feet are feet!
Georgia: Her hubby likes them! LOL!
Lewis: He is all that matters! LOL!
Georgia: OMG, he wrote All of Me was written about! OMG!
Lewis: Her feet! Too hilar for words!
Georgia: I know!
Lewis: So much for having that as a romantic wedding song!
Georgia: LOL! It's about feet!
Lewis: Snerk! Hee!
Georgia: OMG, I'm laughing so hard right now!
Lewis: Me Too!
Georgia: I think, as a couple, those two are so funny together.
Lewis: Yeah, couple goals!
Georgia: For sure!
영어 요약: Lewis and Georgia gossip about Chrissy's feet and Chrissy's relationship.
독어 요약: lewis und georgia klatschen über chrissys füße und chrissys beziehung.
중어 요약: 路易斯和乔治娅在八卦克里斯的脚和感情。


In [11]:
import pandas as pd
import re

# 1. Text cleaning function
def clean_text(text):
    if not isinstance(text, str):
        return text
    text = text.replace('\r', '')          # Remove carriage returns
    text = re.sub(r'\s+', ' ', text)       # Replace multiple spaces with a single space
    return text.strip()                    # Remove leading and trailing spaces

# 2. Integrated preprocessing pipeline function
def preprocess_to_clean_df(dataset_split):
    """
    Takes a Hugging Face Dataset as input, extracts only the 'dialogue' and 'summary' columns,
    and returns a DataFrame with missing values and duplicates removed, and text normalized.
    """
    # Convert to Pandas DataFrame
    df = dataset_split.to_pandas()
    
    # Select only the necessary columns
    df = df[['dialogue', 'summary']].copy()
    
    # Drop missing values
    df = df.dropna(subset=['dialogue', 'summary'])
    
    # Apply text normalization
    df['dialogue'] = df['dialogue'].apply(clean_text)
    df['summary'] = df['summary'].apply(clean_text)
    
    # Remove duplicate data and reset index
    df = df.drop_duplicates().reset_index(drop=True)
    
    return df

# 3. Create DataFrames with updated variable names
train_data = preprocess_to_clean_df(dataset_dict['train'])
val_data = preprocess_to_clean_df(dataset_dict['validation'])
test_data = preprocess_to_clean_df(dataset_dict['test'])

# 4. Check results
print("Preprocessing results!")
print(f"train_data: {train_data.shape}")
print(f"val_data: {val_data.shape}")
print(f"test_data: {test_data.shape}\n")

# Check a sample
display(train_data.head())

Preprocessing results!
train_data: (14731, 2)
val_data: (818, 2)
test_data: (819, 2)



,dialogue,summary
0,Amanda: I baked cookies. Do you want some? Jer...,Amanda baked cookies and will bring Jerry some...
1,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,"Tim: Hi, what's up? Kim: Bad mood tbh, I was g...",Kim may try the pomodoro technique recommended...
3,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,Sam: hey overheard rick say something Sam: i d...,"Sam is confused, because he overheard Rick com..."


In [12]:
train_data['dialogue'].loc[0]

"Amanda: I baked cookies. Do you want some? Jerry: Sure! Amanda: I'll bring you tomorrow :-)"

In [13]:
train_data[['dialogue', 'summary']].head()

,dialogue,summary
0,Amanda: I baked cookies. Do you want some? Jer...,Amanda baked cookies and will bring Jerry some...
1,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,"Tim: Hi, what's up? Kim: Bad mood tbh, I was g...",Kim may try the pomodoro technique recommended...
3,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,Sam: hey overheard rick say something Sam: i d...,"Sam is confused, because he overheard Rick com..."


In [14]:
test_data[['dialogue', 'summary']].head()

,dialogue,summary
0,"Hannah: Hey, do you have Betty's number? Amand...",Hannah needs Betty's number but Amanda doesn't...
1,Eric: MACHINE! Rob: That's so gr8! Eric: I kno...,Eric and Rob are going to watch a stand-up on ...
2,"Lenny: Babe, can you help me with something? B...",Lenny can't decide which trousers to buy. Bob ...
3,"Will: hey babe, what do you want for dinner to...",Emma will be home soon and she will let Will k...
4,"Ollie: Hi , are you in Warsaw Jane: yes, just ...",Jane is in Warsaw. Ollie and Jane has a party....


In [15]:
val_data[['dialogue', 'summary']].head()

,dialogue,summary
0,"A: Hi Tom, are you busy tomorrow’s afternoon? ...",A will go to the animal shelter tomorrow to ge...
1,Emma: I’ve just fallen in love with this adven...,Emma and Rob love the advent calendar. Lauren ...
2,Jackie: Madison is pregnant Jackie: but she do...,Madison is pregnant but she doesn't want to ta...
3,Marla: <file_photo> Marla: look what I found u...,Marla found a pair of boxers under her bed.
4,Robert: Hey give me the address of this music ...,Robert wants Fred to send him the address of t...


In [16]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
from datasets import Dataset, DatasetDict
from transformers import (
    BartTokenizer, 
    BartForConditionalGeneration, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)

# 1. Convert Pandas DataFrames to Hugging Face Dataset format
# Utilizing the previously created train_data, val_data, and test_data.
hg_dataset = DatasetDict({
    "train": Dataset.from_pandas(train_data),
    "validation": Dataset.from_pandas(val_data),
    "test": Dataset.from_pandas(test_data)
})

# 2. Load the Model and Tokenizer
model_name = "facebook/bart-large"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

# 3. Tokenization preprocessing function (applying max_length based on the paper)
def tokenize_function(examples):
    model_inputs = tokenizer(
        examples["dialogue"], 
        max_length=1024, 
        truncation=True
    )
    labels = tokenizer(
        text_target=examples["summary"], 
        max_length=150, 
        truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Map tokenization to the entire dataset (batched=True for speed optimization)
print("start tokenizing")
tokenized_datasets = hg_dataset.map(tokenize_function, batched=True)
print("tokenization complete!")

# 4. Set up Data Collator (Dynamic padding within batches)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# 5. M4 Pro customized training setup (Implementing the paper's batch size of 24)
training_args = Seq2SeqTrainingArguments(
    output_dir="./bart-dialogue-summary-final",
    eval_strategy="epoch",
    learning_rate=2e-5,                  
    per_device_train_batch_size=2,       # Reduced from 8 to 2
    gradient_accumulation_steps=12,      # Increased to 12 (2 * 12 = 24, same as paper)
    per_device_eval_batch_size=2,        # Reduced for evaluation as well
    gradient_checkpointing=True,         # Crucial: Trades compute for memory
    num_train_epochs=20,                 
    weight_decay=0.01,
    save_total_limit=3,
    predict_with_generate=True,
    fp16=False,                          
)

# 6. Initialize Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"], # Add validation set!
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print("Ready to do trainer.train()")

start tokenizing


Map:   0%|          | 0/14731 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

tokenization complete!
Ready to do trainer.train()


In [26]:
# =====================================================================
# STEP 1: SELECT SMALL SUBSET FOR TESTING
# =====================================================================
# Take only the first 500 samples for a quick sanity check
small_train_dataset = tokenized_datasets["train"].select(range(500))
small_eval_dataset = tokenized_datasets["validation"].select(range(100))

# =====================================================================
# STEP 2: MODIFIED TRAINING ARGUMENTS FOR SPEED
# =====================================================================
training_args = Seq2SeqTrainingArguments(
    output_dir="./bart-test-run",
    eval_strategy="no",
    learning_rate=2e-5,
    
    # Increase batch size slightly to speed up M4 Pro (since we have 48GB)
    per_device_train_batch_size=4,       
    gradient_accumulation_steps=6,       # 4 * 6 = 24 (Still effective batch size 24)
    
    # Use max_steps for a quick exit
    max_steps=50,                        # Stop training exactly at 50 steps
    num_train_epochs=1,                  # Ensure it doesn't run more than 1 epoch
    
    optim="adafactor",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    
    logging_steps=10,                    # Log every 10 steps to see progress
    save_strategy="no",                  # Don't waste time saving checkpoints during test
    predict_with_generate=False,
    fp16=False,
    bf16=False,
    report_to="none"
)

# =====================================================================
# STEP 3: INITIALIZE & RUN
# =====================================================================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,   # Use the sliced small dataset
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print(" Starting a high-speed test run (50 steps)...")
trainer.train()

max_steps is given, it will override any value given in num_train_epochs


 Starting a high-speed test run (50 steps)...


  0%|          | 0/50 [00:00<?, ?it/s]

{'loss': 1.4147, 'grad_norm': 13.00816535949707, 'learning_rate': 1.6000000000000003e-05, 'epoch': 0.48}
{'loss': 1.4713, 'grad_norm': 25.074106216430664, 'learning_rate': 1.2e-05, 'epoch': 0.96}
{'loss': 1.3595, 'grad_norm': 16.87904167175293, 'learning_rate': 8.000000000000001e-06, 'epoch': 1.44}
{'loss': 1.444, 'grad_norm': 13.184168815612793, 'learning_rate': 4.000000000000001e-06, 'epoch': 1.92}
{'loss': 1.4742, 'grad_norm': 26.05409049987793, 'learning_rate': 0.0, 'epoch': 2.4}
{'train_runtime': 146.2185, 'train_samples_per_second': 8.207, 'train_steps_per_second': 0.342, 'train_loss': 1.4327547836303711, 'epoch': 2.4}


TrainOutput(global_step=50, training_loss=1.4327547836303711, metrics={'train_runtime': 146.2185, 'train_samples_per_second': 8.207, 'train_steps_per_second': 0.342, 'total_flos': 581122643460096.0, 'train_loss': 1.4327547836303711, 'epoch': 2.4})

In [27]:
# Load ROUGE metric
import evaluate
import numpy as np

rouge = evaluate.load("rouge")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    
    # Decode predictions and labels into text
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    # Replace -100 in the labels as we can't decode them
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Compute ROUGE scores
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v * 100, 4) for k, v in result.items()}

# Run evaluation on the test set
print(" Evaluating on the test set...")
results = trainer.evaluate(eval_dataset=tokenized_datasets["test"], metric_key_prefix="test")
print(results)

 Evaluating on the test set...


  0%|          | 0/103 [00:00<?, ?it/s]

{'test_loss': 1.75719153881073, 'test_runtime': 25.5745, 'test_samples_per_second': 32.024, 'test_steps_per_second': 4.027, 'epoch': 2.4}


In [28]:
def summarize_dialogue(text):
    # 1. Prepare input - this returns a dict containing 'input_ids' and 'attention_mask'
    inputs = tokenizer(text, return_tensors="pt", max_length=512, truncation=True).to(model.device)
    
    # 2. Generate summary - explicitly pass both input_ids and attention_mask
    summary_ids = model.generate(
        input_ids=inputs["input_ids"], 
        attention_mask=inputs["attention_mask"], # Crucial for MPS compatibility
        max_length=150, 
        num_beams=4, 
        early_stopping=True
    )
    
    # 3. Decode and return
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# Test with the sample dialogue again
sample_text = "Yun: Hey, did you finish the training? Yunmin: Yes, it's done! Yun: Great, what's next?"
print(f"Summary: {summarize_dialogue(sample_text)}")

Summary: Yunmin has finished her training.
